# 05 — WO-2 Analytical Investigation## PurposeThis notebook executes the 21 canonical WO-2 SQL investigations in a controlled, evidence-first sequence. It follows the proven WO-1 notebook pattern: environment setup, an explicit readiness check, a separate database-identity/read-only preflight, then one query cell followed by its interpretation and review gate.- Canonical SQL definitions remain in `sql/04_analytics/`.- The shared read-only PostgreSQL runner remains in `src/analytics/db.py`.- This notebook orchestrates query IDs, displays every returned result set, and stores actual outputs in `query_results`.- `notebooks/04_sql_analytics.ipynb` remains the historical WO-1 evidence record and is not modified.## Execution rules1. Run cells individually from top to bottom; **never use Run All**.2. Confirm the preflight output identifies `oulad_analytics` and confirms both read-only settings are `on`.3. Review each query's complete displayed result sets with the project lead, then run its checkpoint cell and type `REVIEWED` when prompted.4. Authorization and review checkpoints use explicit input prompts; no Python source-code edits are required. A checkpoint remains closed unless its required outputs are present, complete, and tied to the current canonical SQL.5. AN3.06 is a high-runtime scan of `staging.student_vle`; run it alone with the explicit 600,000 ms timeout.6. No RAW, QUALITY, or STAGING data is changed. No DDL, index creation, or persistent database modification is performed.**Execution boundary:** the notebook contains no saved WO-2 query outputs. Query execution status is established only by outputs generated in the user's local Jupyter session.


## 1. Environment setup

This cell only locates the repository, imports the shared runner, and initializes the in-memory result dictionary. It does **not** connect to PostgreSQL. A separate following cell confirms that the helper and SQL inventory are available.

In [ ]:
# 1. Resolve the repository root and import the shared read-only query runner.
from pathlib import Path
import sys
from IPython.display import display

_current = Path.cwd().resolve()
_candidates = [_current, *_current.parents]
PROJECT_ROOT = next(
    (
        candidate
        for candidate in _candidates
        if (candidate / "src" / "analytics" / "db.py").is_file()
        and (candidate / "sql" / "04_analytics").is_dir()
    ),
    None,
)

if PROJECT_ROOT is None:
    raise FileNotFoundError(
        "Could not locate the repository root from the current Jupyter directory. "
        "Open Jupyter from the repository root or its notebooks folder."
    )

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.analytics.db import run_query

query_results = {}

print("Project root:", PROJECT_ROOT)
print("Canonical SQL folder:", PROJECT_ROOT / "sql" / "04_analytics")
print("Canonical SQL folder found:", (PROJECT_ROOT / "sql" / "04_analytics").is_dir())
print("Shared runner imported:", callable(run_query))

In [ ]:
# 2. Confirm the notebook environment is ready before opening a database connection.
assert PROJECT_ROOT.is_dir(), "Project root is not available."
assert (PROJECT_ROOT / "sql" / "04_analytics").is_dir(), "Canonical SQL folder is missing."
assert callable(run_query), "The shared query runner was not imported."
assert isinstance(query_results, dict), "The query-results store was not initialized."

print("run_query defined:", "run_query" in globals())
print("query_results defined:", "query_results" in globals())
print("Existing results in memory:", len(query_results))
print("Environment validation: PASS")
print("Database connection: NOT YET OPENED")

**Environment checkpoint:** successful completion of the two preceding cells confirms only that the project files and Python runner are available. It does not prove database connectivity. The next preflight cell opens a separate read-only connection and verifies the actual database identity and session settings.

## 2. Database connection preflight — AN0.01

**Question:** Is the notebook connected to the intended PostgreSQL database and a read-only session?

**Expectation:** The database is `oulad_analytics`; both `transaction_read_only` and `default_transaction_read_only` are `on`.

**Grain:** One row per active database session.

**Source:** PostgreSQL session settings and system functions, through the existing canonical file `sql/04_analytics/AN0.01_database_identity.sql`.

**Safety:** READ-ONLY. The runner opens a separate connection with a read-only default, statement timeout, and lock timeout.

**Runtime:** Very low; metadata only.

**Read as:** Stop if the database name is unexpected or either read-only setting is not `on`. This is a fresh preflight, not a claim that the historical WO-1 output proves the current connection.

In [ ]:
# 3. Open a read-only connection and verify the actual database/session identity.
try:
    preflight = run_query("AN0.01", max_rows=10, statement_timeout_ms=120000)
except Exception as exc:
    print("DATABASE PREFLIGHT: FAILED")
    print("Exception type:", type(exc).__name__)
    print("Exception message:", str(exc))
    raise

query_results["PREFLIGHT.AN0.01"] = preflight
print("Query file:", preflight["query_file"])
print("Elapsed seconds:", preflight["elapsed_seconds"])
print("Result sets:", preflight["result_set_count"])

for item in preflight["result_sets"]:
    print(
        f"Result set {item['statement_number']}: "
        f"{item['rows_displayed']} displayed row(s); "
        f"truncated={item['display_truncated']}"
    )
    display(item["data"])

if preflight["result_set_count"] != 1 or preflight["data"].empty:
    raise RuntimeError("Database preflight returned an unexpected result shape.")

row = preflight["data"].iloc[0]
checks = {
    "database_name": row["database_name"] == "oulad_analytics",
    "transaction_read_only": str(row["transaction_read_only"]).lower() == "on",
    "default_transaction_read_only": str(row["default_transaction_read_only"]).lower() == "on",
}
print("Preflight checks:", checks)

if not all(checks.values()):
    raise RuntimeError(
        "Database preflight failed. Stop here; do not execute WO-2 queries."
    )

PREFLIGHT_VERIFIED = True
print("DATABASE CONNECTION: VERIFIED")
print("READ-ONLY SESSION: VERIFIED")
print("WO-2 query execution may proceed one cell at a time.")

### Preflight interpretation and decision

Complete this checkpoint from the actual output above.

- Confirm the database name is `oulad_analytics`.
- Confirm `transaction_read_only = on`.
- Confirm `default_transaction_read_only = on`.
- Record the displayed statement and lock timeouts.
- If any check fails, stop and resolve the connection configuration before continuing.

**Gate:** `PREFLIGHT_VERIFIED` is set automatically only after all three identity/safety checks pass.

## 3. Controlled execution helperThe next cell defines the WO-2 execution function, authorization prompt, and review-gate helpers. It does not connect to PostgreSQL. The helper:- Requires the successful AN0.01 preflight.- Enforces the named review checkpoints before dependent queries; the user confirms by typing `REVIEWED`, not by editing a Boolean in source code.- Executes one canonical SQL file per call.- Stores the complete bounded result representation in `query_results`.- Displays every result set independently and reports truncation and total batch runtime.- Checks the expected number of result sets for each canonical query.- Prints the exception type and message, then re-raises errors rather than hiding them.The default display cap is 500 rows **per result set**. A truncated display is not a truncated SQL calculation; PostgreSQL executes the query, while only the returned display sample is capped.


In [ ]:
# 4. Controlled query execution, explicit prompts, and dependency-safe review gates.import hashlibLIVE_WO2_EXECUTION_AUTHORIZED = False# Preserve the successful database preflight if this helper cell is re-executed# in the same kernel. A fresh kernel still starts with the safe default False.PREFLIGHT_VERIFIED = globals().get("PREFLIGHT_VERIFIED", False)AN0_07_REVIEWED = FalseAN1_01_REVIEWED = FalseAN1_10_REVIEWED = FalseAN2_10_REVIEWED = FalseAN3_REVIEWED = FalseAN4_D_REVIEWED = FalseAN4_06_REVIEWED = FalseEXPECTED_RESULT_SET_COUNTS = {    "AN0.07": 3,    "AN1.10": 3,    "AN2.10": 3,    "AN3.01": 1,    "AN3.02": 1,    "AN3.03": 1,    "AN3.04": 1,    "AN3.05": 1,    "AN3.06": 1,    "AN3.07": 1,    "AN3.08": 1,    "AN4.01": 1,    "AN4.02": 1,    "AN4.04": 1,    "AN4.05": 2,    "AN4.06": 1,    "AN4.03": 1,    "AN4.07": 1,    "AN4.08": 1,    "AN4.09": 1,    "AN4.10": 1,}# Re-running an upstream query invalidates the review flags and stored outputs# that depend on it. This prevents stale results from satisfying later gates.DEPENDENCY_INVALIDATION = {    "AN0.07": {        "flags": [            "AN1_01_REVIEWED", "AN1_10_REVIEWED", "AN2_10_REVIEWED",            "AN3_REVIEWED", "AN4_D_REVIEWED", "AN4_06_REVIEWED",        ],        "results": [            "PREFLIGHT.AN1.01", "AN1.10", "AN2.10",            "AN3.01", "AN3.02", "AN3.03", "AN3.04",            "AN3.05", "AN3.06", "AN3.07", "AN3.08",            "AN4.01", "AN4.02", "AN4.04", "AN4.05",            "AN4.06", "AN4.03", "AN4.07", "AN4.08",            "AN4.09", "AN4.10",        ],    },    "AN1.10": {        "flags": ["AN2_10_REVIEWED", "AN3_REVIEWED", "AN4_D_REVIEWED", "AN4_06_REVIEWED"],        "results": [            "AN2.10", "AN3.01", "AN3.02", "AN3.03", "AN3.04",            "AN3.05", "AN3.06", "AN3.07", "AN3.08",            "AN4.01", "AN4.02", "AN4.04", "AN4.05",            "AN4.06", "AN4.03", "AN4.07", "AN4.08",            "AN4.09", "AN4.10",        ],    },    "AN2.10": {        "flags": ["AN3_REVIEWED", "AN4_D_REVIEWED", "AN4_06_REVIEWED"],        "results": [            "AN3.01", "AN3.02", "AN3.03", "AN3.04", "AN3.05",            "AN3.06", "AN3.07", "AN3.08",            "AN4.01", "AN4.02", "AN4.04", "AN4.05",            "AN4.06", "AN4.03", "AN4.07", "AN4.08",            "AN4.09", "AN4.10",        ],    },    "AN3.01": {"flags": ["AN3_REVIEWED", "AN4_D_REVIEWED", "AN4_06_REVIEWED"],               "results": ["AN4.01", "AN4.02", "AN4.04", "AN4.05", "AN4.06", "AN4.03", "AN4.07", "AN4.08", "AN4.09", "AN4.10"]},    "AN3.02": {"flags": ["AN3_REVIEWED", "AN4_D_REVIEWED", "AN4_06_REVIEWED"],               "results": ["AN4.01", "AN4.02", "AN4.04", "AN4.05", "AN4.06", "AN4.03", "AN4.07", "AN4.08", "AN4.09", "AN4.10"]},    "AN3.03": {"flags": ["AN3_REVIEWED", "AN4_D_REVIEWED", "AN4_06_REVIEWED"],               "results": ["AN4.01", "AN4.02", "AN4.04", "AN4.05", "AN4.06", "AN4.03", "AN4.07", "AN4.08", "AN4.09", "AN4.10"]},    "AN3.04": {"flags": ["AN3_REVIEWED", "AN4_D_REVIEWED", "AN4_06_REVIEWED"],               "results": ["AN4.01", "AN4.02", "AN4.04", "AN4.05", "AN4.06", "AN4.03", "AN4.07", "AN4.08", "AN4.09", "AN4.10"]},    "AN3.05": {"flags": ["AN3_REVIEWED", "AN4_D_REVIEWED", "AN4_06_REVIEWED"],               "results": ["AN4.01", "AN4.02", "AN4.04", "AN4.05", "AN4.06", "AN4.03", "AN4.07", "AN4.08", "AN4.09", "AN4.10"]},    "AN3.06": {"flags": ["AN3_REVIEWED", "AN4_D_REVIEWED", "AN4_06_REVIEWED"],               "results": ["AN4.01", "AN4.02", "AN4.04", "AN4.05", "AN4.06", "AN4.03", "AN4.07", "AN4.08", "AN4.09", "AN4.10"]},    "AN3.07": {"flags": ["AN3_REVIEWED", "AN4_D_REVIEWED", "AN4_06_REVIEWED"],               "results": ["AN4.01", "AN4.02", "AN4.04", "AN4.05", "AN4.06", "AN4.03", "AN4.07", "AN4.08", "AN4.09", "AN4.10"]},    "AN3.08": {"flags": ["AN3_REVIEWED", "AN4_D_REVIEWED", "AN4_06_REVIEWED"],               "results": ["AN4.01", "AN4.02", "AN4.04", "AN4.05", "AN4.06", "AN4.03", "AN4.07", "AN4.08", "AN4.09", "AN4.10"]},    "AN4.01": {"flags": ["AN4_D_REVIEWED", "AN4_06_REVIEWED"],               "results": ["AN4.06", "AN4.03", "AN4.07", "AN4.08", "AN4.09", "AN4.10"]},    "AN4.02": {"flags": ["AN4_D_REVIEWED", "AN4_06_REVIEWED"],               "results": ["AN4.06", "AN4.03", "AN4.07", "AN4.08", "AN4.09", "AN4.10"]},    "AN4.04": {"flags": ["AN4_D_REVIEWED", "AN4_06_REVIEWED"],               "results": ["AN4.06", "AN4.03", "AN4.07", "AN4.08", "AN4.09", "AN4.10"]},    "AN4.05": {"flags": ["AN4_D_REVIEWED", "AN4_06_REVIEWED"],               "results": ["AN4.06", "AN4.03", "AN4.07", "AN4.08", "AN4.09", "AN4.10"]},    "AN4.06": {"flags": ["AN4_06_REVIEWED"],               "results": ["AN4.03", "AN4.07", "AN4.08", "AN4.09", "AN4.10"]},}def _canonical_sql_path(query_id):    matches = sorted((PROJECT_ROOT / "sql" / "04_analytics").glob(f"{query_id}_*.sql"))    if len(matches) != 1:        raise FileNotFoundError(            f"Expected exactly one canonical SQL file for {query_id}; found {len(matches)}."        )    return matches[0]def request_live_authorization():    if not PREFLIGHT_VERIFIED:        raise RuntimeError(            "The AN0.01 database preflight has not passed. Live execution remains disabled."        )    answer = input(        "Read-only preflight passed. Type AUTHORIZE to enable WO-2 queries for this kernel session: "    )    authorized = answer.strip().upper() == "AUTHORIZE"    print("Live WO-2 execution authorized:", authorized)    return authorizeddef confirm_review(checkpoint_name, query_ids):    if not LIVE_WO2_EXECUTION_AUTHORIZED or not PREFLIGHT_VERIFIED:        raise PermissionError(            "Review cannot be approved until the read-only preflight and live authorization pass."        )    for query_id in query_ids:        result = query_results.get(query_id)        if result is None:            raise RuntimeError(                f"{checkpoint_name}: missing output for {query_id}. Execute it before review."            )        expected = EXPECTED_RESULT_SET_COUNTS[query_id]        if result.get("result_set_count") != expected:            raise RuntimeError(                f"{checkpoint_name}: {query_id} returned an unexpected number of result sets."            )        if len(result.get("result_sets", [])) != expected:            raise RuntimeError(                f"{checkpoint_name}: {query_id} has an incomplete result-set record."            )        if any(item.get("display_truncated", False) for item in result["result_sets"]):            raise RuntimeError(                f"{checkpoint_name}: {query_id} has a truncated result set. "                "Do not approve this checkpoint until the complete output is reviewed."            )        sql_path = _canonical_sql_path(query_id)        current_hash = hashlib.sha256(sql_path.read_bytes()).hexdigest()        if result.get("sql_sha256") != current_hash:            raise RuntimeError(                f"{checkpoint_name}: canonical SQL for {query_id} changed after execution. "                "Re-run that query before approving the checkpoint."            )    if checkpoint_name == "AN0.07":        result_sets = query_results["AN0.07"]["result_sets"]        key_data = result_sets[0]["data"]        expected_tables = {            "assessments", "courses", "student_assessment",            "student_info", "student_registration", "student_vle", "vle",        }        if set(key_data["table_name"]) != expected_tables or len(key_data) != 7:            raise RuntimeError("AN0.07 key inventory differs from the seven expected RAW tables.")        statuses = key_data["key_status"].value_counts().to_dict()        if statuses != {"MATCH": 6, "EXPECTED_NO_PK": 1}:            raise RuntimeError(                f"AN0.07 primary-key statuses require investigation: {statuses}"            )        if not key_data.loc[key_data["key_status"] == "MATCH", "is_validated"].all():            raise RuntimeError("AN0.07 found a primary key that is not validated.")        constraint_data = result_sets[1]["data"]        observed_constraints = dict(            zip(constraint_data["constraint_type"], constraint_data["constraint_count"])        )        expected_constraints = {"c": 0, "f": 0, "p": 6, "u": 0}        if observed_constraints != expected_constraints:            raise RuntimeError(                "AN0.07 constraint inventory differs from the reviewed result: "                f"{observed_constraints}"            )        # The index inventory is intentionally not constrained to be empty:        # any returned index must be inspected, not treated as a failure.    print(f"{checkpoint_name}: all required outputs are present, complete, and tied to current SQL.")    answer = input(        f"After reviewing {checkpoint_name} outputs with the project lead, "        "type REVIEWED to open the next dependency gate: "    )    reviewed = answer.strip().upper() == "REVIEWED"    print(f"{checkpoint_name} reviewed:", reviewed)    if not reviewed:        print("Checkpoint remains closed. No dependent query should be executed.")    return revieweddef execute_query(query_id, *, statement_timeout_ms=120000, requires=None):    if not LIVE_WO2_EXECUTION_AUTHORIZED:        raise PermissionError(            "Live WO-2 execution is disabled. Run the authorization prompt after AN0.01."        )    if not PREFLIGHT_VERIFIED:        raise RuntimeError("Stop gate: the AN0.01 database preflight has not passed.")    required_flags = [requires] if isinstance(requires, str) else (requires or [])    missing_flags = [name for name in required_flags if not globals().get(name, False)]    if missing_flags:        raise RuntimeError(            "Stop gate: run the corresponding review checkpoint cell and enter REVIEWED "            "before continuing. Missing: " + ", ".join(missing_flags)        )    invalidation = DEPENDENCY_INVALIDATION.get(query_id, {"flags": [], "results": []})    for flag in invalidation["flags"]:        globals()[flag] = False    for stale_query_id in invalidation["results"]:        query_results.pop(stale_query_id, None)    query_results.pop(query_id, None)    sql_path = _canonical_sql_path(query_id)    sql_hash_before = hashlib.sha256(sql_path.read_bytes()).hexdigest()    try:        result = run_query(            query_id,            max_rows=500,            statement_timeout_ms=statement_timeout_ms,        )    except Exception as exc:        print(f"{query_id}: EXECUTION FAILED")        print("Exception type:", type(exc).__name__)        print("Exception message:", str(exc))        raise    sql_hash_after = hashlib.sha256(sql_path.read_bytes()).hexdigest()    if sql_hash_after != sql_hash_before:        raise RuntimeError(            f"{query_id}: canonical SQL changed during execution. "            "Discard this output and rerun after reviewing the file."        )    result["sql_sha256"] = sql_hash_before    query_results[query_id] = result    print(f"{query_id}: EXECUTION RETURNED")    print("Query file:", result["query_file"])    print("Result sets returned:", result["result_set_count"])    print("Batch elapsed seconds:", result["elapsed_seconds"])    print("Statement timeout (ms):", result["statement_timeout_ms"])    print("Display row cap per result set:", result["max_rows"])    for item in result["result_sets"]:        status = "TRUNCATED" if item["display_truncated"] else "complete display"        print(            f"Result set {item['statement_number']}: "            f"{item['rows_displayed']} row(s) displayed; {status}"        )        display(item["data"])    expected = EXPECTED_RESULT_SET_COUNTS[query_id]    if result["result_set_count"] != expected:        raise RuntimeError(            f"{query_id} returned {result['result_set_count']} result set(s); "            f"{expected} were expected from the canonical SQL file. "            "Review the displayed outputs and runner behavior before proceeding."        )    print(f"{query_id}: RESULT-SET COUNT CHECK PASS ({expected})")    return result


### Explicit live-execution authorizationThe live-execution switch remains disabled by default. After the successful AN0.01 preflight, run the next cell and type `AUTHORIZE` at the prompt to enable read-only WO-2 queries for the current kernel session. No source-code edit is required. Any kernel restart resets authorization.


In [ ]:
# Explicit authorization prompt — no source-code edit is required.LIVE_WO2_EXECUTION_AUTHORIZED = request_live_authorization()


In [ ]:
# 5. Confirm the execution helper and review gates are available.
assert callable(execute_query), "execute_query was not defined."
assert isinstance(EXPECTED_RESULT_SET_COUNTS, dict)
assert len(EXPECTED_RESULT_SET_COUNTS) == 21
assert isinstance(query_results, dict)

print("execute_query defined:", callable(execute_query))
print("Expected WO-2 query IDs:", len(EXPECTED_RESULT_SET_COUNTS))
print("Result storage ready:", isinstance(query_results, dict))
print("Preflight verified:", PREFLIGHT_VERIFIED)
print("WO-2 execution helper validation: PASS")

## 4. WO2-A — Structural verification and join fan-out

The sequence is AN0.07 → review → current AN1.01 key refresh → review → AN1.10. The current AN1.01 refresh is a dependency check using the established canonical WO-1 SQL; it is not counted among the 21 WO-2 investigations.

### AN0.07 — RAW primary-key and index verification

**Question:** Do the primary-key constraints that actually exist in the RAW schema match the keys documented in `docs/data_dictionary.md`?

**Expectation:** Verify whether six RAW tables have validated primary keys on exactly the documented columns, in the documented order. Inspect the actual constraints and all indexes on `raw.student_vle` without assuming that the absence of a primary key implies the absence of indexes.

**Grain:** Statement 1: one row per RAW table; statement 2: one row per constraint type; statement 3: one row per index on `raw.student_vle`.

**Source:** PostgreSQL catalogs `pg_constraint`, `pg_class`, `pg_namespace`, `pg_attribute`, and `pg_indexes`.

**Safety:** READ-ONLY; catalog metadata only.

**Runtime:** Very low.

**Read as:** `key_status` must be `MATCH` for six tables and `EXPECTED_NO_PK` for `student_vle`. Any other value is an observation to investigate, not a reason to alter RAW. This check does not establish uniqueness of `student_vle` rows; that question is handled under AN5.

**Review gate:** Inspect all three result sets before proceeding.

In [ ]:
an0_07 = execute_query("AN0.07", requires="PREFLIGHT_VERIFIED")

### Interpretation and decision — AN0.07Record the actual result-set count, each RAW table's key status, constraint counts, and the complete index inventory for `raw.student_vle`. Do not infer uniqueness from index presence or absence. Do not modify RAW.**Review checkpoint:** After the three result sets have been interpreted, run the following checkpoint cell and type `REVIEWED` at the prompt. The cell also verifies the seven-table key inventory, the six validated primary keys, the expected constraint counts, complete result displays, and that the canonical SQL has not changed since execution.


In [ ]:
# Human review prompt — type REVIEWED only after the three outputs have been interpreted.AN0_07_REVIEWED = confirm_review("AN0.07", ["AN0.07"])


### AN1.01 — Current student-presentation key refresh (dependency check)

**Question:** Is `(code_module, code_presentation, id_student)` complete and unique in both `student_info` and `student_registration` in the current database state?

**Expectation:** No NULL key components and no duplicate key groups in either table.

**Grain:** One row per source table.

**Source:** `staging.student_info` and `staging.student_registration`.

**Safety:** READ-ONLY.

**Runtime:** Low.

**Read as:** Both rows must report `PASS`, zero NULL key rows, and zero duplicate-key groups. This fresh check protects the grain assumption used by AN1.10.

In [ ]:
if not AN0_07_REVIEWED:    raise RuntimeError(        "AN0.07 is not reviewed. Run its checkpoint cell and type REVIEWED first."    )# A fresh AN1.01 run invalidates any previously approved downstream results.AN1_01_REVIEWED = Falsefor flag in [    "AN1_10_REVIEWED", "AN2_10_REVIEWED", "AN3_REVIEWED",    "AN4_D_REVIEWED", "AN4_06_REVIEWED",]:    globals()[flag] = Falsefor stale_query_id in [    "AN1.10", "AN2.10", "AN3.01", "AN3.02", "AN3.03", "AN3.04",    "AN3.05", "AN3.06", "AN3.07", "AN3.08", "AN4.01", "AN4.02",    "AN4.04", "AN4.05", "AN4.06", "AN4.03", "AN4.07", "AN4.08",    "AN4.09", "AN4.10",]:    query_results.pop(stale_query_id, None)sql_path = _canonical_sql_path("AN1.01")sql_hash_before = hashlib.sha256(sql_path.read_bytes()).hexdigest()an1_01_refresh = run_query("AN1.01", max_rows=20, statement_timeout_ms=120000)if hashlib.sha256(sql_path.read_bytes()).hexdigest() != sql_hash_before:    raise RuntimeError("AN1.01 SQL changed during execution; discard and rerun.")an1_01_refresh["sql_sha256"] = sql_hash_beforequery_results["PREFLIGHT.AN1.01"] = an1_01_refreshprint("Query file:", an1_01_refresh["query_file"])print("Elapsed seconds:", an1_01_refresh["elapsed_seconds"])print("Result sets:", an1_01_refresh["result_set_count"])display(an1_01_refresh["data"])if (    an1_01_refresh["result_set_count"] != 1    or an1_01_refresh["display_truncated"]    or an1_01_refresh["data"].empty):    raise RuntimeError("AN1.01 dependency refresh returned an incomplete or unexpected result.")key_check = an1_01_refresh["data"]expected_tables = {"student_info", "student_registration"}if len(key_check) != 2 or set(key_check["table_name"]) != expected_tables:    raise RuntimeError(        "AN1.01 must return exactly one row for student_info and one for student_registration."    )if not (key_check["gate_status"].astype(str).str.upper() == "PASS").all():    raise RuntimeError("AN1.01 key-grain dependency failed. Stop before AN1.10.")if (    (key_check["null_key_rows"] != 0).any()    or (key_check["duplicate_key_groups"] != 0).any()    or (key_check["excess_rows"] != 0).any()):    raise RuntimeError("AN1.01 key completeness/uniqueness checks failed. Stop before AN1.10.")AN1_01_REVIEWED = Trueprint("AN1.01 current key-grain dependency: PASS (two expected tables; zero null/duplicate/excess rows).")


### Interpretation and decision — AN1.01 dependency refresh

Confirm both source tables pass the current composite-key test. If either fails, stop; AN1.10's fan-out decomposition depends on the student-presentation grain. This refresh is separate from the WO-2 query inventory.

### AN1.10 — Assessment join fan-out decomposition

**Question:** Which students and presentations explain the 33,407-row increase in the `id_student`-only join from AN1.03?

**Expectation:** The excess equals `SUM(submissions * (presentations_of_student - 1))` and comes only from students registered in more than one presentation.

**Grain:** Statement 1: one row per presentation count of the student; statement 2: one row per module and presentation; statement 3: one reconciliation row.

**Source:** `staging.student_assessment`, `staging.assessments`, `staging.student_info`.

**Safety:** READ-ONLY.

**Runtime:** Low to moderate.

**Read as:** The fan-out is explained only if `reconciles = TRUE`, `submissions_without_student_info = 0`, and `unmatched_on_correct_path = 0`. A different total from the historical baseline is a finding to investigate, not an error. Never join `student_assessment` to `student_info` on `id_student` alone.

**Review gate:** Inspect all three result sets and require reconciliation before proceeding.

In [ ]:
an1_10 = execute_query(
    "AN1.10",
    requires=["AN0_07_REVIEWED", "AN1_01_REVIEWED"],
)

### Interpretation and decision — AN1.10Reconcile the fan-out formula, presentation-level decomposition, submissions without student information, and unmatched rows on the correct join path. Compare the observed excess with the historical 33,407-row reference without forcing a match.**Review checkpoint:** After all three result sets have been reconciled with the project lead, run the following cell and type `REVIEWED` at the prompt.


In [ ]:
# Human review prompt — type REVIEWED only after all three AN1.10 outputs have been reconciled.AN1_10_REVIEWED = confirm_review("AN1.10", ["AN1.10"])


## 5. WO2-B — IMD missingness and initial withdrawal investigation

### AN2.10 — IMD missingness extension

**Question:** Is missing `imd_band` concentrated by module or region-by-presentation, and can it be recovered from the same person in another presentation?

**Expectation:** Historical North Region (40.10%) and Ireland (22.47%) concentrations are hypotheses until checked. Some missing rows may have alternative values elsewhere.

**Grain:** Statement 1: module; statement 2: every region/module/presentation group with at least one missing IMD value; statement 3: recovery class (rows and distinct students).

**Source:** `staging.student_info`.

**Safety:** READ-ONLY.

**Runtime:** Low.

**Read as:** Reconcile missing rows to the historical 1,111-row reference. A value in another presentation does not prove the IMD of this presentation; conflicting values must not be resolved by picking one. No imputation is approved.

In [ ]:
an2_10 = execute_query("AN2.10", requires="AN1_10_REVIEWED")

### Interpretation and decision — AN2.10Record missing-row and distinct-student denominators by module and region/presentation. Reconcile against the historical 1,111-row reference. Separate same-person alternative values from conflicting alternatives. Do not impute or infer a missingness mechanism.**Review checkpoint:** After inspecting all three result sets with the project lead, run the following cell and type `REVIEWED` at the prompt.


In [ ]:
# Human review prompt — type REVIEWED only after all three AN2.10 outputs have been interpreted.AN2_10_REVIEWED = confirm_review("AN2.10", ["AN2.10"])


## 6. WO2-C — Outcomes, withdrawal, and registration timing

Execute AN3.01–AN3.08 individually. AN3.06 is isolated as a high-runtime scan and must not be combined with other queries.

### AN3.01 — Final result versus unregistration

**Question:** How consistent is final_result = 'Withdrawn' with the presence of date_unregistration?

**Expectation:** Historical counts (10,072 rows with an unregistration date and 10,156 Withdrawn rows) are reconciliation references only; the current cross-tab must be read without forcing those values.

**Grain:** One row per final_result and unregistration-date presence.

**Source:** staging.student_info joined to staging.student_registration on the full three-column key.

**Safety:** READ-ONLY.

**Runtime:** Low.

**Read as:** The join is one-to-one (AN1.01). Non-zero off-diagonal cells are real data conflicts: do not define withdrawal from one field until they are explained.

**Execution note:** Inspect both off-diagonal populations; do not treat final_result and unregistration date as interchangeable.

In [ ]:
an3_01 = execute_query(
    "AN3.01",
    requires="AN2_10_REVIEWED",
)

### Interpretation and decision — AN3.01

Record the actual counts, denominators, time basis, missing-value treatment, and limitations from the returned output. Do not infer causation or change source values.

### AN3.02 — Withdrawal definition conflicts

**Question:** Which rows break the equivalence Withdrawn <=> has date_unregistration, and how large is each conflict group by presentation?

**Expectation:** Potential conflict groups are Withdrawn without a date and non-Withdrawn with a date; their prevalence and timing must be established from the returned output.

**Grain:** One row per conflict group, module, and presentation.

**Source:** staging.student_info, staging.student_registration, staging.courses.

**Safety:** READ-ONLY.

**Runtime:** Low.

**Read as:** NULL-safe comparison is used although final_result is NOT NULL in RAW. Do not label any group erroneous without context. Compare date_unregistration with module_presentation_length for the non-Withdrawn group. A date after the length suggests a post-course action rather than a withdrawal.

**Execution note:** Inspect conflict counts and timing relative to presentation length before defining any withdrawal rule.

In [ ]:
an3_02 = execute_query(
    "AN3.02",
    requires="AN2_10_REVIEWED",
)

### Interpretation and decision — AN3.02

Record the actual counts, denominators, time basis, missing-value treatment, and limitations from the returned output. Do not infer causation or change source values.

### AN3.03 — Withdrawal timing distribution

**Question:** When during a presentation do students unregister?

**Expectation:** Historical timing patterns are hypotheses only; no bucket prevalence or module difference is assumed.

**Grain:** One row per module, presentation, and timing bucket.

**Source:** staging.student_registration with staging.courses.

**Safety:** READ-ONLY.

**Runtime:** Low.

**Read as:** Buckets are relative to presentation start (day 0). Negative days mean before start. Percentages use all unregistrations of that presentation as the denominator, not all registrations.

**Execution note:** The after-course-end condition takes precedence over in-course week buckets; inspect all timing categories.

In [ ]:
an3_03 = execute_query(
    "AN3.03",
    requires="AN2_10_REVIEWED",
)

### Interpretation and decision — AN3.03

Record the actual counts, denominators, time basis, missing-value treatment, and limitations from the returned output. Do not infer causation or change source values.

### AN3.04 — Registration timing profile

**Question:** How early do students register, and does registration timing relate to withdrawal?

**Expectation:** Historical observations (including 45 missing registration dates) are reference hypotheses only; verify current bucket counts and outcome rates.

**Grain:** One row per registration-timing bucket.

**Source:** staging.student_registration and staging.student_info.

**Safety:** READ-ONLY.

**Runtime:** Low.

**Read as:** Report the NULL bucket separately; never drop it silently. Association only: earlier registration may reflect motivation, not cause outcomes.

**Execution note:** Retain missing registration dates and interpret withdrawal differences as descriptive associations, not causal effects.

In [ ]:
an3_04 = execute_query(
    "AN3.04",
    requires="AN2_10_REVIEWED",
)

### Interpretation and decision — AN3.04

Record the actual counts, denominators, time basis, missing-value treatment, and limitations from the returned output. Do not infer causation or change source values.

### AN3.05 — Registration and unregistration order

**Question:** Are there impossible or edge-case date orderings (unregistered before registering, same day, before start)?

**Expectation:** No ordering pattern is assumed; historical counts are reference points only and must not determine classification.

**Grain:** One row per ordering category.

**Source:** staging.student_registration.

**Safety:** READ-ONLY.

**Runtime:** Low.

**Read as:** BEFORE_REGISTRATION rows are data-quality candidates. SAME_DAY rows are legitimate but must be kept out of 'time enrolled' averages (zero duration).

**Execution note:** Classify missing, reversed, same-day, and later date sequences without changing source values.

In [ ]:
an3_05 = execute_query(
    "AN3.05",
    requires="AN2_10_REVIEWED",
)

### Interpretation and decision — AN3.05

Record the actual counts, denominators, time basis, missing-value treatment, and limitations from the returned output. Do not infer causation or change source values.

### AN3.06 — Withdrawn student-presentations and recorded VLE activity

**Question:** Among student-presentations classified as Withdrawn, how many have a recorded VLE activity row, and how does the last recorded activity day compare with the unregistration day?

**Expectation:** The query describes recorded-row presence and timing. No direction or prevalence is assumed.

**Grain:** One row per module, presentation, and recorded-activity group among Withdrawn student-presentations.

**Source:** staging.student_info, staging.student_registration, staging.student_vle (aggregated to student-presentation first).

**Safety:** READ-ONLY.

**Runtime:** HIGH. Scans staging.student_vle (historically about 10.6M rows). Run alone. Use the runner's explicit 600,000 ms timeout; do not add SET commands.

**Read as:** NO_ACTIVITY means no matching row was found in staging.student_vle for this student-presentation. It does not prove the student never clicked, did not learn, or used no other channel. Report both the Withdrawn denominator and the full student-presentation denominator. Do not use sum_click.

**Execution note:** Run this cell alone. Review the denominator, registration-match classification, and recorded-activity categories before proceeding.

In [ ]:
an3_06 = execute_query(
    "AN3.06",
    statement_timeout_ms=600000,
    requires="AN2_10_REVIEWED",
)

### Interpretation and decision — AN3.06

Record the actual counts, denominators, time basis, missing-value treatment, and limitations from the returned output. Do not infer causation or change source values.

### AN3.07 — Outcome rates by previous attempts and education

**Question:** How do final_result shares vary with num_of_prev_attempts and highest_education?

**Expectation:** No directional association is assumed. Small cells (n < 100) require cautious interpretation.

**Grain:** One row per dimension, category, and final_result.

**Source:** staging.student_info.

**Safety:** READ-ONLY.

**Runtime:** Low.

**Read as:** Shares use the category total as the denominator. This is a student-presentation grain: repeat students appear once per presentation, so rows are not independent people. Association, not causation.

**Execution note:** Inspect category denominators and flag small cells before interpreting outcome shares.

In [ ]:
an3_07 = execute_query(
    "AN3.07",
    requires="AN2_10_REVIEWED",
)

### Interpretation and decision — AN3.07

Record the actual counts, denominators, time basis, missing-value treatment, and limitations from the returned output. Do not infer causation or change source values.

### AN3.08 — Withdrawal rate by presentation

**Question:** Which module-presentations have unusually high or low withdrawal?

**Expectation:** No withdrawal-rate range or presentation ordering is assumed; all differences are descriptive.

**Grain:** One row per module and presentation.

**Source:** staging.student_info.

**Safety:** READ-ONLY.

**Runtime:** Low.

**Read as:** Differences between presentations can reflect cohort size and module design, not quality. Check the denominator before ranking.

**Execution note:** Use the student-presentation denominator; do not turn descriptive rates into a presentation-quality ranking.

In [ ]:
an3_08 = execute_query(
    "AN3.08",
    requires="AN2_10_REVIEWED",
)

### WO2-C interpretation and decisionAfter AN3.01–AN3.08 have all been executed, record:- The operational distinction between `final_result` and `date_unregistration`.- Date-order and post-course timing anomalies.- Denominators for registration timing and withdrawal shares.- The exact meaning of recorded VLE activity absence in AN3.06.- Small-cell limitations and repeated student appearances across presentations.**Review checkpoint:** After all eight AN3 outputs have been interpreted, run the following cell and type `REVIEWED` at the prompt. The gate verifies that every required output is present, complete, and tied to its current SQL file.


In [ ]:
# Human review prompt — all eight AN3 outputs must be present and complete.AN3_REVIEWED = confirm_review(    "WO2-C",    ["AN3.01", "AN3.02", "AN3.03", "AN3.04", "AN3.05", "AN3.06", "AN3.07", "AN3.08"],)


## 7. WO2-D — Assessment structure, dates, and scores

Run AN4.01, AN4.02, AN4.04, and AN4.05 before interpreting banked status. AN4.03 is intentionally deferred until after AN4.06.

### AN4.01 — Assessment structure by presentation

**Question:** What assessments does each presentation contain, and how do weights add up?

**Expectation:** No assessment-type pattern or weight-total rule is assumed; inspect the observed structure by presentation.

**Grain:** One row per module, presentation, and assessment type.

**Source:** staging.assessments.

**Safety:** READ-ONLY.

**Runtime:** Low.

**Read as:** Weight totals that differ from 100 are a finding, not an error, until the dataset documentation is checked. Do not average scores across types before reading this table.

In [ ]:
an4_01 = execute_query(
    "AN4.01",
    requires="AN3_REVIEWED",
)

### Interpretation and decision — AN4.01

Record assessment counts separately from submission-row counts, preserve NULLs, keep numeric zero distinct from NULL, and retain every observed assessment type and banked-status value.

### AN4.02 — Assessments with missing dates

**Question:** Which assessment records have no recorded due date, and what are their observed types and submission counts?

**Expectation:** No assessment type or presentation pattern is assumed. The historical count of 11 is a reconciliation reference only.

**Grain:** One row per assessment with a NULL due date.

**Source:** staging.assessments and staging.student_assessment.

**Safety:** READ-ONLY.

**Runtime:** Low.

**Read as:** Treat assessment_type as observed data, not as a precondition. Reconcile the number of assessments separately from the number of linked submission rows. Preserve any NULL assessment_type value.

In [ ]:
an4_02 = execute_query(
    "AN4.02",
    requires="AN3_REVIEWED",
)

### Interpretation and decision — AN4.02

Record assessment counts separately from submission-row counts, preserve NULLs, keep numeric zero distinct from NULL, and retain every observed assessment type and banked-status value.

### AN4.04 — Submissions without a due date

**Question:** How many submission rows are linked to assessments without a recorded due date, and how are they distributed?

**Expectation:** No assessment type, module, presentation, or shared grouping pattern is assumed. The historical total of 2,865 is a reconciliation reference only.

**Grain:** One row per module, presentation, assessment type, and banked-record status; include distinct assessment count.

**Source:** staging.student_assessment and staging.assessments.

**Safety:** READ-ONLY.

**Runtime:** Low.

**Read as:** Reconcile the total submission rows with AN4.02. Do not infer that all rows share one assessment type. Preserve NULL/other banked status. A missing due date prevents direct due-date timing calculation; do not substitute course end.

In [ ]:
an4_04 = execute_query(
    "AN4.04",
    requires="AN3_REVIEWED",
)

### Interpretation and decision — AN4.04

Record assessment counts separately from submission-row counts, preserve NULLs, keep numeric zero distinct from NULL, and retain every observed assessment type and banked-status value.

### AN4.05 — Score profile and NULL scores

**Question:** What is the score distribution, including NULL and zero scores, across assessment types and banked-record statuses?

**Expectation:** No score distribution or banked-status pattern is assumed. The historical 173 NULL-score rows are a reconciliation reference only.

**Grain:** Statement 1: one row per assessment type and banked status; statement 2: one row per assessment with NULL scores and banked status.

**Source:** staging.student_assessment and staging.assessments.

**Safety:** READ-ONLY.

**Runtime:** Low.

**Read as:** A NULL score means not recorded, not zero. Keep zero scores separate. Preserve all banked statuses and do not exclude records before AN4.06 is interpreted.

In [ ]:
an4_05 = execute_query(
    "AN4.05",
    requires="AN3_REVIEWED",
)

### Interpretation and decision — AN4.05

Record assessment counts separately from submission-row counts, preserve NULLs, keep numeric zero distinct from NULL, and retain every observed assessment type and banked-status value.

### WO2-D review checkpointReconcile assessment counts, submission counts, missing due dates, score distributions, NULL scores, and zero scores. Document the observed types and banked-status representations.**Review checkpoint:** After reviewing all four WO2-D outputs, run the following cell and type `REVIEWED` at the prompt.


In [ ]:
# Human review prompt — all four WO2-D outputs must be present and complete.AN4_D_REVIEWED = confirm_review(    "WO2-D",    ["AN4.01", "AN4.02", "AN4.04", "AN4.05"],)


## 8. WO2-E — Banked assessments, timing, participation, and coverage

**Required order:** AN4.06 first. Review the stored banked flag before executing AN4.03 and the remaining banked-status analyses. Banked status remains a descriptive dimension; no exclusion is approved by this notebook.

### AN4.06 — Banked assessment profile (run first)

**Question:** How are student_assessment records distributed across observed is_banked values and assessment types?

**Expectation:** No prevalence, origin, validity, or timing pattern is assumed.

**Grain:** One row per raw is_banked value (including NULL) and assessment type.

**Source:** staging.student_assessment and staging.assessments.

**Safety:** READ-ONLY.

**Runtime:** Low.

**Read as:** This is a descriptive profile of the stored flag. It does not establish why a record is banked or whether it should be excluded. Review this query before AN4.03. Preserve all statuses in downstream profiles until a separate evidence-based decision is approved.

In [ ]:
an4_06 = execute_query(
    "AN4.06",
    requires="AN4_D_REVIEWED",
)

### Interpretation and decision — AN4.06Record all observed values of `is_banked`, including NULL, by assessment type. The profile does not establish why records are banked or whether they should be excluded.**Review checkpoint:** After interpreting the complete AN4.06 output, run the following cell and type `REVIEWED` at the prompt.


In [ ]:
# Human review prompt — type REVIEWED only after interpreting the AN4.06 banked-status profile.AN4_06_REVIEWED = confirm_review("AN4.06", ["AN4.06"])


### AN4.03 — Submission timing versus due date

**Question:** How early or late are submissions relative to the recorded assessment due date, and how does timing vary by banked-record status?

**Expectation:** No timing distribution or banked-status prevalence is assumed.

**Grain:** One row per assessment type, banked-record status, and timing category.

**Source:** staging.student_assessment joined to staging.assessments on id_assessment.

**Safety:** READ-ONLY.

**Runtime:** Moderate.

**Read as:** days_late = date_submitted - due date. Assessments with NULL due dates are profiled separately in AN4.04. Keep is_banked as a dimension, including NULL/other values. Keep missing submission dates as a separate timing category. AN4.06 must be reviewed first; do not infer that banked rows are invalid or exclude them automatically.

In [ ]:
an4_03 = execute_query(
    "AN4.03",
    requires="AN4_06_REVIEWED",
)

### Interpretation and decision — AN4.03

Record actual counts, denominators, banked-status overlap, missing values, and timing semantics. Keep the result descriptive; do not infer accepted submissions or causal effects from date comparisons.

### AN4.07 — Assessment participation by outcome

**Question:** How many assessment submission rows are recorded per student-presentation, by final_result and banked-record status, including students with no submission row?

**Expectation:** No outcome ordering or banked-status prevalence is assumed.

**Grain:** One row per final_result, banked status (including no-submission), and submission-count bucket; student-presentation is the counting unit.

**Source:** staging.student_info, staging.assessments, staging.student_assessment.

**Safety:** READ-ONLY.

**Runtime:** Moderate.

**Read as:** Submission counts are grouped by the assessment's own module-presentation and student. A student-presentation may appear in more than one banked-status group if its records carry more than one status. Zero-submission student-presentations are explicitly retained.

In [ ]:
an4_07 = execute_query(
    "AN4.07",
    requires="AN4_06_REVIEWED",
)

### Interpretation and decision — AN4.07

Record actual counts, denominators, banked-status overlap, missing values, and timing semantics. Keep the result descriptive; do not infer accepted submissions or causal effects from date comparisons.

### AN4.08 — Score by final result

**Question:** How do recorded non-NULL assessment scores vary across final_result, assessment type, and banked-record status?

**Expectation:** No monotonic score pattern is assumed.

**Grain:** One row per final_result, assessment type, and banked status.

**Source:** staging.student_info, staging.assessments, staging.student_assessment.

**Safety:** READ-ONLY.

**Runtime:** Moderate.

**Read as:** Scores and final_result are associated and may be mechanically related, especially for exams. This is descriptive only and is not a valid feature set for predicting final_result. The distinct count is at student-presentation grain, not unique-person grain across the full dataset. Preserve all banked statuses.

In [ ]:
an4_08 = execute_query(
    "AN4.08",
    requires="AN4_06_REVIEWED",
)

### Interpretation and decision — AN4.08

Record actual counts, denominators, banked-status overlap, missing values, and timing semantics. Keep the result descriptive; do not infer accepted submissions or causal effects from date comparisons.

### AN4.09 — Submissions after unregistration

**Question:** How do recorded submission dates compare with the registration record's unregistration day, by banked status?

**Expectation:** No timing prevalence is assumed.

**Grain:** One row per module, presentation, assessment type, banked status, and timing category.

**Source:** staging.student_assessment, staging.assessments, staging.student_registration.

**Safety:** READ-ONLY.

**Runtime:** Moderate.

**Read as:** This is a date comparison, not proof that a submission was accepted after withdrawal. Preserve all banked statuses. Keep missing unregistration dates, missing submission dates, and missing registration matches separate. Do not report only the after-unregistration category.

In [ ]:
an4_09 = execute_query(
    "AN4.09",
    requires="AN4_06_REVIEWED",
)

### Interpretation and decision — AN4.09

Record actual counts, denominators, banked-status overlap, missing values, and timing semantics. Keep the result descriptive; do not infer accepted submissions or causal effects from date comparisons.

### AN4.10 — Weighted-score grain and coverage check

**Question:** How much assessment weight is represented by recorded submission rows per student-presentation, and how does coverage vary by banked status?

**Expectation:** No coverage level is assumed. The output tests whether a weighted score would have comparable assessment-weight coverage.

**Grain:** One row per module, presentation, and banked status; student-presentation is the unit used to calculate each student's coverage.

**Source:** staging.student_info, staging.assessments, staging.student_assessment.

**Safety:** READ-ONLY.

**Runtime:** Moderate.

**Read as:** Weight is counted once per student-assessment-status combination after collapsing repeated physical rows at that combination. This is a metric sensitivity diagnostic, not a final weighted-score definition. A student may occur in more than one status group. Do not infer score validity from coverage alone.

In [ ]:
an4_10 = execute_query(
    "AN4.10",
    requires="AN4_06_REVIEWED",
)

### Interpretation and decision — AN4.10

Confirm the output grain and the treatment of repeated physical submission rows. The average and median submitted-weight shares use student-presentations with the observed status as their denominator, not the full presentation population. The full student-presentation count is context only. A student may occur in more than one banked-status group.

**Important:** No `zero_weight_rows` diagnostic is included. This query does not approve a weighted-score KPI, a final-score rule, or automatic banked-record exclusion.

## 9. WO2-F — Evidence review and closeout

No execution status is inferred from the existence of a query cell. The in-memory `query_results` dictionary contains only query outputs returned in the current kernel session.

For each query, retain and review:
- Execution status, runtime, result-set count, displayed row counts, and truncation.
- Output grain, join cardinality, unmatched rows, and denominator.
- Missing-value treatment and time basis.
- Distinct student versus student-presentation counts.
- Banked-status treatment and overlap between groups.
- Material limitations, alternative explanations, and evidence-based decisions.

| Query ID | Execution verified? | Key result / denominator | Interpretation and limitation | Decision / follow-up |
|---|---|---|---|---|
| AN0.07–AN4.10 | Not established by notebook preparation | | | |

**Exit rule:** WO-2 is not complete until outputs have been reviewed and reconciled, or explicitly deferred with a documented reason. WO-2 does not authorize AN5, KPI approval, analytical marts, Power BI, predictive modelling, or the separate data-science phase.